# 1. Imports

In [13]:
import pandas as pd

from dataclasses import dataclass
from typing_extensions import NotRequired

from pydantic import BaseModel

from langchain.agents import create_agent, AgentState
from langchain.messages import ToolMessage
from langchain.tools import tool, ToolRuntime
from langchain_ollama import ChatOllama
from langchain_experimental.tools import PythonAstREPLTool

from langgraph.types import Command

# 2. Configure Ollama

In [36]:
MODEL_NAME = "gemma4:e4b"

llm = ChatOllama(
    model=MODEL_NAME,
    temperature=0,
)


ScalarValue = int | float | str | bool | None


class DataAnalysisAnswer(BaseModel):
    summary: str
    result: dict[str, ScalarValue]


class AnalysisSummary(BaseModel):
    summary: str


@dataclass
class DataFrameContext:
    data: pd.DataFrame


class DataFrameState(AgentState):
    result: NotRequired[dict[str, ScalarValue]]


summary_llm = llm.with_structured_output(
    AnalysisSummary,
    method="json_schema",
)

# 3. Create DataFrame

In [19]:
DATA_PATH = "../_task_artidis_nlp_assignment/physical_exam_study.csv"

df = pd.read_csv(DATA_PATH)

df.head()

,gender,intensity,pain_score
0,m,4.6,6
1,f,6.0,5
2,f,5.5,3
3,m,4.1,6
4,m,3.6,5


## Print dataframe info

In [20]:
print("Shape:", df.shape)
print()
print("Columns:")
print(df.dtypes)
print()
print("Summary statistics:", df.describe(include="all"))
print("Genders:", df["gender"].value_counts())
print("Missing values:")
print(df.isna().sum())

Shape: (2000, 3)

Columns:
gender         object
intensity     float64
pain_score      int64
dtype: object

Summary statistics:        gender    intensity   pain_score
count    2000  2000.000000  2000.000000
unique      2          NaN          NaN
top         m          NaN          NaN
freq     1000          NaN          NaN
mean      NaN     4.165700     4.153500
std       NaN     1.097413     2.984022
min       NaN     0.200000     0.000000
25%       NaN     3.400000     1.000000
50%       NaN     4.200000     4.000000
75%       NaN     4.900000     7.000000
max       NaN     8.300000    10.000000
Genders: gender
m    1000
f    1000
Name: count, dtype: int64
Missing values:
gender        0
intensity     0
pain_score    0
dtype: int64


## Dataframe execution tool

In [41]:
import ast


@tool
def execute_dataframe_code(
    query: str,
    runtime: ToolRuntime[DataFrameContext, DataFrameState],
) -> Command:
    """
    Execute Python/Pandas code against the current dataframe.

    The dataframe is available as `df`.
    """

    repl = PythonAstREPLTool(
        locals={
            "df": runtime.context.data,
            "pd": pd,
        }
    )

    output = repl.invoke(query)

    # Preferred case: model created a structured result.
    result = repl.locals.get("result")

    if isinstance(result, dict):
        result = {
            str(key): value.item() if hasattr(value, "item") else value
            for key, value in result.items()
        }

    else:
        # Fallback: preserve a successful scalar tool result.
        try:
            value = ast.literal_eval(str(output).strip())
        except (ValueError, SyntaxError):
            value = str(output).strip()

        if hasattr(value, "item"):
            value = value.item()

        result = {"value": value}

    return Command(
        update={
            "result": result,
            "messages": [
                ToolMessage(
                    content=str(result),
                    tool_call_id=runtime.tool_call_id,
                )
            ],
        }
    )

# 4. Build the agent

In [60]:
AGENT_PROMPT = """
You are a data analysis agent working with a pandas DataFrame named `df`.

You MUST use the execute_dataframe_code tool for every dataframe calculation.
Never estimate numeric answers yourself.

IMPORTANT:
- Prefer exactly ONE tool call per user question.
- Calculate ALL requested values in that single call.
- For questions requesting multiple values, assign them all to a
  dictionary named `result`.
- Do not call the tool separately for each statistic.
- Do not repeat a calculation that has already succeeded.

Use standard Python values with float() or int().

Example for multiple values:

values = df["temperature"]

result = {
    "minimum_temperature": float(values.min()),
    "maximum_temperature": float(values.max()),
}

Example for one value:

result = {
    "mean_temperature": float(df["temperature"].mean()),
}
"""


agent = create_agent(
    model=llm,
    tools=[execute_dataframe_code],
    system_prompt=AGENT_PROMPT,
    context_schema=DataFrameContext,
    state_schema=DataFrameState,
)

# 6. Retrieval queries

## 1. Setup ask-answer functions

In [61]:
import time


def ask(
    question: str,
    data: pd.DataFrame,
) -> DataAnalysisAnswer:

    print(f"\n{'=' * 80}")
    print(f"Question: {question}")
    print("=" * 80)

    context = DataFrameContext(data=data)

    dataframe_info = f"""
DataFrame shape: {data.shape}

Columns:
{data.dtypes.to_string()}

Sample rows:
{data.head(2).to_string(index=False)}

Question:
{question}
"""

    start = time.perf_counter()

    agent_result = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": dataframe_info,
                }
            ]
        },
        context=context,
    )

    agent_time = time.perf_counter() - start

    computed_result = agent_result.get("result")

    if computed_result is None:
        raise ValueError(
            "The agent did not produce a structured dataframe result."
        )

    start = time.perf_counter()

    summary_result = summary_llm.invoke(
        f"""
Question:
{question}

Computed result:
{computed_result}

Explain the result briefly.
Use only the supplied values.
Do not claim statistical significance unless it was explicitly calculated.
"""
    )

    summary_time = time.perf_counter() - start

    final_result = DataAnalysisAnswer(
        summary=summary_result.summary,
        result=computed_result,
    )

    print("\nComputed result:")
    print(computed_result)

    print(
        f"\nTiming: agent={agent_time:.2f}s, "
        f"summary={summary_time:.2f}s"
    )

    print("\nFinal answer:")
    print(final_result)

    return final_result

In [62]:
result_1 = ask(
    "What is the mean and standard deviation "
    "of the workout intensity for men?",
    df,
)

result_2 = ask(
    "What are the minimum and maximum values "
    "of the pain score for women?",
    df,
)

result_3 = ask(
    "Is there a correlation between workout intensity "
    "and perceived pain? Calculate the Pearson correlation "
    "for the complete dataset and separately for men and women. "
    "Does the correlation differ between men and women?",
    df,
)


Question: What is the mean and standard deviation of the workout intensity for men?

Computed result:
{'mean_intensity_men': 4.2542, 'std_intensity_men': 1.0731607680278077}

Timing: agent=4.03s, summary=3.19s

Final answer:
summary='The average (mean) workout intensity for men is 4.2542. The standard deviation of 1.0732 indicates that the typical workout intensity for men deviates from this mean by approximately 1.0732.' result={'mean_intensity_men': 4.2542, 'std_intensity_men': 1.0731607680278077}

Question: What are the minimum and maximum values of the pain score for women?

Computed result:
{'minimum_pain_score_women': 0.0, 'maximum_pain_score_women': 7.0}

Timing: agent=3.08s, summary=2.63s

Final answer:
summary='The minimum pain score observed for women is 0.0, and the maximum pain score observed for women is 7.0.' result={'minimum_pain_score_women': 0.0, 'maximum_pain_score_women': 7.0}

Question: Is there a correlation between workout intensity and perceived pain? Calculate 

In [63]:
import math


def numeric_values(answer: DataAnalysisAnswer) -> list[float]:
    return sorted(
        float(value)
        for value in answer.result.values()
        if isinstance(value, (int, float))
    )


def assert_values(
    answer: DataAnalysisAnswer,
    expected: list[float],
    tolerance: float = 1e-6,
):
    actual = numeric_values(answer)
    expected = sorted(expected)

    assert len(actual) == len(expected), (
        f"Expected {len(expected)} values, got {len(actual)}: "
        f"{answer.result}"
    )

    for actual_value, expected_value in zip(actual, expected):
        assert math.isclose(
            actual_value,
            expected_value,
            rel_tol=tolerance,
            abs_tol=tolerance,
        ), f"Expected {expected_value}, got {actual_value}"

    print("✓ PASS")

In [64]:
test_df = pd.DataFrame({
    "city": ["Zagreb", "Berlin", "London", "Paris", "Rome"],
    "temperature": [28.0, 24.0, 21.0, 26.0, 29.0],
})

answer = ask(
    "What is the mean temperature?",
    test_df,
)

assert_values(
    answer,
    [25.6],
)


Question: What is the mean temperature?

Computed result:
{'mean_temperature': 25.6}

Timing: agent=1.96s, summary=2.48s

Final answer:
summary='The mean temperature is 25.6.' result={'mean_temperature': 25.6}
✓ PASS


In [65]:
answer = ask(
    "What are the minimum and maximum temperatures?",
    test_df,
)

assert_values(
    answer,
    [21.0, 29.0],
)


Question: What are the minimum and maximum temperatures?

Computed result:
{'minimum_temperature': 21.0, 'maximum_temperature': 29.0}

Timing: agent=2.31s, summary=2.55s

Final answer:
summary='The minimum temperature is 21.0, and the maximum temperature is 29.0.' result={'minimum_temperature': 21.0, 'maximum_temperature': 29.0}
✓ PASS


In [66]:
answer = ask(
    "How many rows are in the dataframe?",
    test_df,
)

assert_values(
    answer,
    [5],
)


Question: How many rows are in the dataframe?

Computed result:
{'value': 5}

Timing: agent=1.99s, summary=2.34s

Final answer:
summary='The dataframe contains 5 rows.' result={'value': 5}
✓ PASS


In [67]:
sales_df = pd.DataFrame({
    "product": ["A", "B", "C", "D"],
    "sales": [100, 200, 300, 400],
})

answer = ask(
    "What is the total sales value?",
    sales_df,
)

assert_values(
    answer,
    [1000],
)


Question: What is the total sales value?

Computed result:
{'total_sales': 1000.0}

Timing: agent=2.20s, summary=2.49s

Final answer:
summary='The total sales value is 1000.0.' result={'total_sales': 1000.0}
✓ PASS


In [68]:
df_a = pd.DataFrame({
    "value": [1, 2, 3],
})

df_b = pd.DataFrame({
    "value": [100, 200, 300],
})

answer_a = ask(
    "What is the mean value?",
    df_a,
)

answer_b = ask(
    "What is the mean value?",
    df_b,
)


assert_values(answer_a, [2.0])
assert_values(answer_b, [200.0])


Question: What is the mean value?

Computed result:
{'mean_value': 2.0}

Timing: agent=1.96s, summary=2.68s

Final answer:
summary='The mean value is 2.0.' result={'mean_value': 2.0}

Question: What is the mean value?

Computed result:
{'mean_value': 200.0}

Timing: agent=2.07s, summary=2.80s

Final answer:
summary='The mean value is 200.0.' result={'mean_value': 200.0}
✓ PASS
✓ PASS


In [69]:
people_df = pd.DataFrame({
    "group": ["a", "a", "b", "b"],
    "score": [10, 20, 100, 200],
})

answer = ask(
    "What is the mean score for group b?",
    people_df,
)

assert_values(
    answer,
    [150.0],
)


Question: What is the mean score for group b?

Computed result:
{'mean_score_group_b': 150.0}

Timing: agent=2.46s, summary=2.46s

Final answer:
summary='The mean score for group b is 150.0.' result={'mean_score_group_b': 150.0}
✓ PASS


In [70]:
answer = ask(
    "Calculate the mean score separately for group a and group b.",
    people_df,
)

assert_values(
    answer,
    [15.0, 150.0],
)


Question: Calculate the mean score separately for group a and group b.

Computed result:
{'mean_score_group_a': 15.0, 'mean_score_group_b': 150.0}

Timing: agent=3.77s, summary=3.54s

Final answer:
summary='The mean score for Group A is 15.0, while the mean score for Group B is 150.0. Group B has a substantially higher mean score than Group A.' result={'mean_score_group_a': 15.0, 'mean_score_group_b': 150.0}
✓ PASS


In [71]:
answer = ask(
    "What is the mean and standard deviation "
    "of the workout intensity for men?",
    df,
)

expected_mean = df.loc[
    df["gender"] == "m",
    "intensity",
].mean()

expected_std = df.loc[
    df["gender"] == "m",
    "intensity",
].std()

assert_values(
    answer,
    [expected_mean, expected_std],
)


Question: What is the mean and standard deviation of the workout intensity for men?

Computed result:
{'mean_intensity_men': 4.2542, 'std_intensity_men': 1.0731607680278077}

Timing: agent=3.88s, summary=3.41s

Final answer:
summary='The average (mean) workout intensity for men is 4.2542. The standard deviation of 1.0732 indicates that the typical workout intensity for men deviates from this mean by approximately 1.0732.' result={'mean_intensity_men': 4.2542, 'std_intensity_men': 1.0731607680278077}
✓ PASS


In [72]:
answer = ask(
    "What are the minimum and maximum values "
    "of the pain score for women?",
    df,
)

expected_min = df.loc[
    df["gender"] == "f",
    "pain_score",
].min()

expected_max = df.loc[
    df["gender"] == "f",
    "pain_score",
].max()

assert_values(
    answer,
    [expected_min, expected_max],
)


Question: What are the minimum and maximum values of the pain score for women?

Computed result:
{'minimum_pain_score_women': 0.0, 'maximum_pain_score_women': 7.0}

Timing: agent=3.29s, summary=2.87s

Final answer:
summary='The minimum pain score observed for women is 0.0, and the maximum pain score observed for women is 7.0.' result={'minimum_pain_score_women': 0.0, 'maximum_pain_score_women': 7.0}
✓ PASS


In [73]:
answer = ask(
    "Calculate the Pearson correlation between intensity and pain score "
    "for the complete dataset, for men, and for women.",
    df,
)

overall = df["intensity"].corr(
    df["pain_score"]
)

men = (
    df.loc[df["gender"] == "m", "intensity"]
    .corr(df.loc[df["gender"] == "m", "pain_score"])
)

women = (
    df.loc[df["gender"] == "f", "intensity"]
    .corr(df.loc[df["gender"] == "f", "pain_score"])
)

assert_values(
    answer,
    [overall, men, women],
)


Question: Calculate the Pearson correlation between intensity and pain score for the complete dataset, for men, and for women.

Computed result:
{'overall_correlation': 0.5032174582559054, 'men_correlation': 0.8390746332687103, 'women_correlation': 0.8163610031517332}

Timing: agent=5.81s, summary=5.78s

Final answer:
summary='The Pearson correlation between intensity and pain score is positive across all groups, indicating that generally, higher intensity is associated with higher pain scores. The overall correlation is 0.5032. When analyzing gender separately, both men (0.8391) and women (0.8164) show a stronger positive correlation than the overall dataset, suggesting a more pronounced linear relationship between intensity and pain score within each gender group.' result={'overall_correlation': 0.5032174582559054, 'men_correlation': 0.8390746332687103, 'women_correlation': 0.8163610031517332}
✓ PASS
